# Experiment 1 — MNIST-Addition with ABLkit

**TRAP formula:** `f(g(x), x)` — symbolic abduction corrects the perceptual layer.  
**Week 2, Days 4–5**

## Task
Reproduce the MNIST-Addition benchmark using ABLkit, then extend it:  
- Compare **LeNet** (default) vs **ResNet-18** perceptual backbones  
- Vary label noise at 0%, 10%, 20%, 30%  
- Log the number of abduction loops to convergence at each noise level  
- Plot test accuracy vs. (loops × noise level)  

## References
- Zhou 2019 — Abductive Learning, *Science China Inf. Sci.* 62:76101  
- Huang et al. 2024 — ABLkit, *Frontiers of Computer Science* 18(6):186354  
- ABLkit docs: https://ablkit.readthedocs.io/
- ABLkit GitHub: https://github.com/AbductiveLearning/ABLkit

## 0 · Environment check

In [ ]:
import sys, importlib

required = ['torch', 'torchvision', 'ablkit', 'numpy', 'matplotlib']
for pkg in required:
    try:
        importlib.import_module(pkg)
        print(f'  ✓  {pkg}')
    except ImportError:
        print(f'  ✗  {pkg}  ← run: pip install {pkg}')

import torch
print(f'\nPyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}')

## 1 · Imports

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

# ABLkit
from ablkit.learning import ABLModel
from ablkit.reasoning import GroundKB
from ablkit.bridge import SimpleBridge
from ablkit.data.evaluation import ReasoningMetric

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_ROOT = Path('../../data')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Device: {DEVICE}')

## 2 · MNIST-Addition dataset

Each sample is a pair of MNIST digit images; the label is their integer sum (0–18).  
We inject label noise by randomly flipping `noise_rate` fraction of sum-labels.

In [ ]:
def make_mnist_addition_dataset(root, split='train', noise_rate=0.0):
    """Return lists of (img1, img2, digit1, digit2, noisy_sum) tuples."""
    is_train = split == 'train'
    mnist = torchvision.datasets.MNIST(
        root=root, train=is_train, download=True,
        transform=T.Compose([T.ToTensor(), T.Normalize((0.1307,), (0.3081,))])
    )
    # group indices by digit
    by_digit = {d: [] for d in range(10)}
    for idx, (_, lbl) in enumerate(mnist):
        by_digit[lbl].append(idx)

    pairs = []
    rng = np.random.default_rng(SEED)
    n_pairs = 30000 if is_train else 5000
    for _ in range(n_pairs):
        d1, d2 = rng.integers(0, 10, size=2)
        idx1 = rng.choice(by_digit[int(d1)])
        idx2 = rng.choice(by_digit[int(d2)])
        img1 = mnist[idx1][0]
        img2 = mnist[idx2][0]
        true_sum = int(d1) + int(d2)
        noisy_sum = true_sum
        if noise_rate > 0 and rng.random() < noise_rate:
            noisy_sum = rng.integers(0, 19)
        pairs.append((img1, img2, int(d1), int(d2), noisy_sum))
    return pairs

# Quick sanity check
sample = make_mnist_addition_dataset(DATA_ROOT, split='train', noise_rate=0.1)
img1, img2, d1, d2, s = sample[0]
print(f'Sample: digit1={d1}, digit2={d2}, noisy_sum={s}')
print(f'Dataset size: {len(sample)}')

## 3 · Perceptual models

We compare two backbones.  
**LeNet-5**: lightweight, the ABLkit default.  
**ResNet-18**: larger; we replace the final layer for 10-class digit classification.

In [ ]:
class LeNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, 5, padding=2), nn.Tanh(),
            nn.AvgPool2d(2),
            nn.Conv2d(6, 16, 5), nn.Tanh(),
            nn.AvgPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 5 * 5, 120), nn.Tanh(),
            nn.Linear(120, 84), nn.Tanh(),
            nn.Linear(84, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def make_resnet18(num_classes=10):
    model = torchvision.models.resnet18(weights=None)
    # MNIST is 1-channel 28×28; patch first conv and remove maxpool
    model.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


print('LeNet params:', sum(p.numel() for p in LeNet().parameters()))
print('ResNet-18 params:', sum(p.numel() for p in make_resnet18().parameters()))

## 4 · Knowledge base (abductive bridge)

The KB encodes the constraint: `digit1 + digit2 = observed_sum`.  
ABLkit's abduction loop finds consistent pseudo-label pairs (d1, d2).

In [ ]:
# TODO: implement using ABLkit's GroundKB / KBBase API.
# Consult: https://ablkit.readthedocs.io/en/latest/Tutorials/MNIST_Addition.html
#
# Sketch of the required interface:
#
# class AdditionKB(GroundKB):
#     def pseudo_label_list(self, model_outs, candidates):
#         """Return consistent (d1, d2) pairs for each (img1, img2, sum) triple."""
#         ...
#
# Alternatively, use the built-in MNIST-Addition example from ABLkit:
#   from examples.mnist_add import MnistAdditionKB
print('TODO: implement AdditionKB')

## 5 · Training loop

The ABL loop alternates:  
1. Neural training step (update f with current pseudo-labels)  
2. Abduction step (revise pseudo-labels using KB consistency check)

In [ ]:
def run_experiment(backbone_name: str, noise_rate: float, max_loops: int = 20):
    """
    Train an ABL model for MNIST-Addition.

    Returns:
        dict with keys: 'test_acc', 'loops_to_converge', 'acc_per_loop'
    """
    print(f'\n── backbone={backbone_name}, noise={noise_rate:.0%} ──')

    # 1. build perceptual model
    if backbone_name == 'lenet':
        base_model = LeNet(num_classes=10).to(DEVICE)
    else:
        base_model = make_resnet18(num_classes=10).to(DEVICE)

    # 2. wrap in ABLModel
    # TODO: replace with real ABLkit ABLModel wrapper
    # abl_model = ABLModel(base_model, ...)

    # 3. build KB
    # kb = AdditionKB()

    # 4. build bridge
    # bridge = SimpleBridge(abl_model, kb)

    # 5. load data
    train_data = make_mnist_addition_dataset(DATA_ROOT, 'train', noise_rate)
    test_data  = make_mnist_addition_dataset(DATA_ROOT, 'test',  0.0)

    # 6. run ABL loop
    acc_per_loop = []
    for loop in range(max_loops):
        # bridge.train(train_data)  # one round of neural + abductive updates
        # acc = bridge.eval(test_data)
        acc = 0.0  # placeholder
        acc_per_loop.append(acc)
        print(f'  Loop {loop+1:02d}  test_acc={acc:.4f}')

        # early stopping (convergence criterion)
        if loop > 2 and abs(acc_per_loop[-1] - acc_per_loop[-2]) < 1e-4:
            break

    return {
        'test_acc': acc_per_loop[-1],
        'loops_to_converge': len(acc_per_loop),
        'acc_per_loop': acc_per_loop,
    }

print('run_experiment() defined — run the grid in the next cell.')

## 6 · Experiment grid

2 backbones × 4 noise levels = 8 runs.

In [ ]:
BACKBONES = ['lenet', 'resnet18']
NOISE_RATES = [0.0, 0.1, 0.2, 0.3]

results = {}
for bb in BACKBONES:
    for nr in NOISE_RATES:
        key = (bb, nr)
        results[key] = run_experiment(bb, nr)

print('\nGrid complete.')

## 7 · Plot: test accuracy vs. (loops × noise level)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
cmap = plt.cm.viridis

for ax_idx, bb in enumerate(BACKBONES):
    ax = axes[ax_idx]
    for nr in NOISE_RATES:
        r = results[(bb, nr)]
        xs = range(1, len(r['acc_per_loop']) + 1)
        ax.plot(xs, r['acc_per_loop'], marker='o', label=f'noise={nr:.0%}')
    ax.set_title(f'Backbone: {bb}')
    ax.set_xlabel('Abduction loop')
    ax.set_ylabel('Test accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle('MNIST-Addition (ABLkit): accuracy vs. loops × noise level')
plt.tight_layout()
plt.savefig('mnist_addition_results.png', dpi=150)
plt.show()

## 8 · Results table

In [ ]:
import pandas as pd

rows = []
for (bb, nr), r in results.items():
    rows.append({
        'backbone': bb,
        'noise_rate': f'{nr:.0%}',
        'test_acc': f"{r['test_acc']:.4f}",
        'loops_to_converge': r['loops_to_converge'],
    })

pd.DataFrame(rows).pivot(index='noise_rate', columns='backbone', values=['test_acc', 'loops_to_converge'])

## 9 · TRAP annotation

**TRAP formula instantiated:** `f(g(x), x)`

- `f` is the digit classifier (LeNet or ResNet-18).  
- `g` is the ABL knowledge base: it takes model outputs and the KB constraint (`d1 + d2 = sum`) and returns *revised* pseudo-labels by abduction.  
- The loop `f(g(x), x)` corresponds to TRAP's **Perception** component: the symbolic meta-function `g` corrects the perceptual model `f` through consistency-based abduction.  

**Observation from experiment:**  
*(fill in after running the grid)*  
- Which backbone converges faster?  
- How does noise rate affect the number of loops needed?  
- At what noise level does accuracy degrade significantly?